<a href="https://colab.research.google.com/github/alizamca2025114-tech/Android-Layout-Demo/blob/main/RailDrishti.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas gradio google-genai

In [ ]:
import pandas as pd
import sqlite3
import gradio as gr
from datetime import datetime

print("RailDrishti setup initialized successfully!")

RailDrishti setup initialized successfully!


In [ ]:
# Create Railway Administration Database

conn = sqlite3.connect("railway_admin.db")
cursor = conn.cursor()

print("Railway Administration database connected!")

Railway Administration database connected!


In [ ]:
# RAILDRISHTI DATABASE STRUCTURE

# 1. Stations
cursor.execute("""
CREATE TABLE IF NOT EXISTS stations (
    station_code TEXT PRIMARY KEY,
    station_name TEXT NOT NULL,
    zone TEXT,
    division TEXT)
""")

# 2. Trains
cursor.execute("""
CREATE TABLE IF NOT EXISTS trains (
    train_number TEXT PRIMARY KEY,
    train_name TEXT NOT NULL,
    source TEXT,
    destination TEXT,
    departure_time TEXT,
    arrival_time TEXT,
    status TEXT)
""")

# 3. Railway Assets
cursor.execute("""
CREATE TABLE IF NOT EXISTS assets (
    asset_id TEXT PRIMARY KEY,
    asset_type TEXT NOT NULL,
    location TEXT,
    condition TEXT,
    criticality INTEGER,
    last_inspection TEXT)
""")

# 4. Defects
cursor.execute("""
CREATE TABLE IF NOT EXISTS defects (
    defect_id TEXT PRIMARY KEY,
    asset_id TEXT,
    defect_type TEXT,
    severity TEXT,
    reported_date TEXT,
    status TEXT)
""")

# 5. Maintenance Block Requests
cursor.execute("""
CREATE TABLE IF NOT EXISTS block_requests (
    block_id TEXT PRIMARY KEY,
    section TEXT,
    start_time TEXT,
    end_time TEXT,
    purpose TEXT,
    requested_by TEXT,
    status TEXT)
""")

# 6. Staff / Resources
cursor.execute("""
CREATE TABLE IF NOT EXISTS staff (
    staff_id TEXT PRIMARY KEY,
    name TEXT,
    department TEXT,
    availability TEXT,
    assigned_location TEXT)
""")

conn.commit()

print("✓ Stations table created")
print("✓ Trains table created")
print("✓ Assets table created")
print("✓ Defects table created")
print("✓ Block Requests table created")
print("✓ Staff table created")
print("\nRailDrishti database structure ready!")

✓ Stations table created
✓ Trains table created
✓ Assets table created
✓ Defects table created
✓ Block Requests table created
✓ Staff table created

RailDrishti database structure ready!


In [ ]:
# RAILDRISHTI - STATION DATA

stations = [
    ("CSMT", "Chhatrapati Shivaji Maharaj Terminus", "Central Railway", "Mumbai"),
    ("DR", "Dadar", "Central Railway", "Mumbai"),
    ("TNA", "Thane", "Central Railway", "Mumbai"),
    ("KYN", "Kalyan", "Central Railway", "Mumbai"),
    ("LTT", "Lokmanya Tilak Terminus", "Central Railway", "Mumbai"),
    ("PNVL", "Panvel", "Central Railway", "Mumbai")]

cursor.executemany("""
INSERT OR REPLACE INTO stations
(station_code, station_name, zone, division)
VALUES (?, ?, ?, ?)
""", stations)

conn.commit()

print(f"✓ {len(stations)} stations added")

✓ 6 stations added


In [ ]:
# RAILDRISHTI - TRAIN DATA

trains = [
    ("12951", "Mumbai Rajdhani", "Mumbai Central",
     "New Delhi", "17:00", "08:35", "Scheduled"),

    ("12110", "Panchavati Express", "Manmad",
     "CSMT", "06:10", "10:25", "Scheduled"),

    ("11010", "Sinhagad Express", "Pune",
     "CSMT", "06:05", "09:45", "Scheduled"),

    ("12124", "Deccan Queen", "Pune",
     "CSMT", "07:15", "09:25", "Scheduled"),

    ("22152", "Tejas Express", "CSMT",
     "Karmali", "05:50", "14:00", "Scheduled"),

    ("11009", "Deccan Express", "CSMT",
     "Pune", "17:10", "21:05", "Scheduled")
]

cursor.executemany("""
INSERT OR REPLACE INTO trains
(train_number, train_name, source, destination,
 departure_time, arrival_time, status)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", trains)

conn.commit()

print(f"✓ {len(trains)} trains added")

✓ 6 trains added


In [ ]:
# Display railway stations

station_data = pd.read_sql_query(
    "SELECT * FROM stations",
    conn
)

display(station_data)

,station_code,station_name,zone,division
0,CSMT,Chhatrapati Shivaji Maharaj Terminus,Central Railway,Mumbai
1,DR,Dadar,Central Railway,Mumbai
2,TNA,Thane,Central Railway,Mumbai
3,KYN,Kalyan,Central Railway,Mumbai
4,LTT,Lokmanya Tilak Terminus,Central Railway,Mumbai
5,PNVL,Panvel,Central Railway,Mumbai


In [ ]:
# Display trains

train_data = pd.read_sql_query(
    "SELECT * FROM trains",
    conn
)

display(train_data)

,train_number,train_name,source,destination,departure_time,arrival_time,status
0,12951,Mumbai Rajdhani,Mumbai Central,New Delhi,17:00,08:35,Scheduled
1,12110,Panchavati Express,Manmad,CSMT,06:10,10:25,Scheduled
2,11010,Sinhagad Express,Pune,CSMT,06:05,09:45,Scheduled
3,12124,Deccan Queen,Pune,CSMT,07:15,09:25,Scheduled
4,22152,Tejas Express,CSMT,Karmali,05:50,14:00,Scheduled
5,11009,Deccan Express,CSMT,Pune,17:10,21:05,Scheduled


In [ ]:
# RAILDRISHTI - RAILWAY ASSET DATA

assets = [
    ("AST001", "Track", "CSMT-Dadar Section",
     "Good", 6, "2026-09-01"),

    ("AST002", "Track", "Dadar-Thane Section",
     "Needs Inspection", 9, "2026-08-20"),

    ("AST003", "Signal", "Thane Yard",
     "Good", 7, "2026-09-03"),

    ("AST004", "Point Machine", "Thane Yard",
     "Critical", 10, "2026-08-15"),

    ("AST005", "Bridge", "Kalyan Section",
     "Needs Maintenance", 9, "2026-08-10"),

    ("AST006", "Overhead Equipment", "Thane-Kalyan Section",
     "Good", 7, "2026-09-02"),

    ("AST007", "Signal", "Kalyan Station",
     "Needs Inspection", 8, "2026-08-25"),

    ("AST008", "Track", "Kalyan-Panvel Section",
     "Good", 5, "2026-09-04")
]

cursor.executemany("""
INSERT OR REPLACE INTO assets
(asset_id, asset_type, location, condition,
 criticality, last_inspection)
VALUES (?, ?, ?, ?, ?, ?)
""", assets)

conn.commit()

print(f"✓ {len(assets)} railway assets added")

✓ 8 railway assets added


In [ ]:
asset_data = pd.read_sql_query(
    "SELECT * FROM assets",
    conn)

display(asset_data)

,asset_id,asset_type,location,condition,criticality,last_inspection
0,AST001,Track,CSMT-Dadar Section,Good,6,2026-09-01
1,AST002,Track,Dadar-Thane Section,Needs Inspection,9,2026-08-20
2,AST003,Signal,Thane Yard,Good,7,2026-09-03
3,AST004,Point Machine,Thane Yard,Critical,10,2026-08-15
4,AST005,Bridge,Kalyan Section,Needs Maintenance,9,2026-08-10
5,AST006,Overhead Equipment,Thane-Kalyan Section,Good,7,2026-09-02
6,AST007,Signal,Kalyan Station,Needs Inspection,8,2026-08-25
7,AST008,Track,Kalyan-Panvel Section,Good,5,2026-09-04
